# Sometria — dataset analysis

Reproduces the checks behind the preprocessing decisions:
resample to 60 Hz, crop to 5 s with duration-weighted sampling, exclude corrupt-torque samples.

Run `preprocess()`, then `splits.py` before this notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch as t
import yaml
from scipy import signal

from sometria.preprocess import _load_sample
from sometria.representation import _load_human_definition

ROOT = Path("../data/processed")
RAW_ROOT = Path("/home/z1ko/datasets/sometria/hpc_v2_with_babel_actions")
CACHE = ROOT / "psd_cache.npz"
HUMAN = _load_human_definition("../config/human.yaml")
DOFS = HUMAN["dofs"]
CHANNELS = ["pos", "vel", "acc", "tau"]
TARGET_HZ = 60.0          # every stored sample is already at this rate
WINDOW_S = 5.0            # proposed crop length

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
C = {"pos": "#4878a8", "vel": "#6acc64", "acc": "#ee854a", "tau": "#d1615d"}

df = pl.read_parquet(ROOT / "samples.parquet").with_columns(
    pl.col("path").str.split("/").list.get(0).alias("dataset"))
print(f"{len(df)} samples, {df['duration'].sum()/3600:.1f} h, {df['dataset'].n_unique()} datasets")
df.head(3)

## 1. Durations

The spikes are not motion structure — they are the upstream 2000-frame cap seen at each source rate
(2000/120 = 16.7 s, 2000/100 = 20 s, 2000/60 = 33.3 s).

In [ ]:
d = df["duration"].to_numpy()
# The 2000-frame cap is upstream, so it has to be measured in source frames: resampling to
# 60 Hz turned a capped 120 Hz clip into 1000 frames.
capped = (df["n_frames"] * df["original_hz"] / df["hz"] >= 1999).to_numpy()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
bins = np.arange(0, 35, 0.5)
ax[0].hist([d[~capped], d[capped]], bins=bins, stacked=True, color=["#4878a8", "#d1615d"],
           label=[f"full ({(~capped).sum()})", f"at 2000-frame cap ({capped.sum()})"])
ax[0].set(xlabel="duration (s)", ylabel="samples", title="Sample durations")
ax[0].legend(fontsize=8)

ax[1].ecdf(d, color="#4878a8")
for x in (2, WINDOW_S, 10):
    ax[1].axvline(x, ls=":", c="grey", lw=1)
    ax[1].annotate(f"{x:g}s → {100*(d<=x).mean():.0f}%", (x, 0.06), fontsize=8,
                   rotation=90, color="grey", ha="right")
ax[1].set(xlabel="duration (s)", ylabel="fraction ≤ x", title="Cumulative", xlim=(0, 35))

top = df.group_by("dataset").len().sort("len", descending=True).head(10)["dataset"].to_list()
ax[2].boxplot([df.filter(pl.col("dataset") == k)["duration"].to_numpy() for k in top],
              tick_labels=top, orientation="horizontal", showfliers=False, widths=.6)
ax[2].set(xlabel="duration (s)", title="By dataset (top 10)")
ax[2].invert_yaxis()
fig.tight_layout()

print(f"median {np.median(d):.1f}s   p90 {np.percentile(d,90):.1f}s   max {d.max():.1f}s")
print(f"at cap: {capped.sum()} ({100*capped.mean():.1f}%)   uncapped median {np.median(d[~capped]):.1f}s")

## 2. Sample rates

Three clocks (120 / 100 / 60 Hz), each internally uniform. This is *multi-rate*, not *irregular* —
which is why resampling solves it and irregular-time-series machinery is not needed.

In [ ]:
rate = df["original_hz"].round(0)   # stored `hz` is uniformly 60 after preprocessing
vc = df.group_by(rate.alias("hz_r")).len().sort("len", descending=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar([str(int(r)) for r in vc["hz_r"][:6]], vc["len"][:6], color="#4878a8")
ax[0].set(xlabel="nominal source Hz", ylabel="samples", title="Source sample rate")
for i, v in enumerate(vc["len"][:6]):
    ax[0].text(i, v, f" {v}", ha="center", va="bottom", fontsize=8)

jit = []
for r in df.sample(200, seed=5).iter_rows(named=True):
    tt = pl.read_csv(RAW_ROOT / r["path"], columns=["time"])["time"].to_numpy()
    dt = np.diff(tt)
    jit.append(dt.std() / dt.mean() * 100)
ax[1].hist(jit, bins=40, color="#6acc64")
ax[1].set(xlabel="within-sample dt coefficient of variation (%)", ylabel="samples",
          title=f"Source sampling regularity (median {np.median(jit):.3f}%)")
fig.tight_layout()

## 3. Spectral content — can we resample to 60 Hz?

60 Hz keeps everything below 30 Hz. The question is how much power lives above that line.
First pass over every sample with Nyquist ≥ 50 Hz; cached to `psd_cache.npz`.

In [ ]:
FGRID = np.linspace(0, 50, 201)

def scan_psd():
    # frame count is in stored 60 Hz frames; welch needs a decent window in *source* frames
    sel = df.filter((pl.col("original_hz") >= 99)
                    & (pl.col("n_frames") * pl.col("original_hz") / pl.col("hz") >= 512))
    acc_curve = {c: np.zeros(len(FGRID)) for c in CHANNELS}
    above = {c: [] for c in CHANNELS}
    paths = []
    for i, r in enumerate(sel.iter_rows(named=True), 1):
        m = _load_sample(RAW_ROOT / r["path"], HUMAN)["motion"]
        for ci, c in enumerate(CHANNELS):
            x = m[:, :, ci].astype(np.float64)
            x = x - x.mean(0)
            f, P = signal.welch(x, fs=r["original_hz"], nperseg=min(512, len(x)), axis=0)
            P = P.sum(1)
            tot = P.sum()
            above[c].append(float(P[f > 30].sum() / tot) if tot > 0 else 0.0)
            acc_curve[c] += np.interp(FGRID, f, P / tot if tot > 0 else P)
        paths.append(r["path"])
        if i % 4000 == 0:
            print(f"  {i}/{len(sel)}")
    n = len(paths)
    np.savez(CACHE, fgrid=FGRID, n=n, paths=np.array(paths),
             **{f"curve_{c}": acc_curve[c] / n for c in CHANNELS},
             **{f"above_{c}": np.array(above[c]) for c in CHANNELS})

if not CACHE.exists() or int(np.load(CACHE, allow_pickle=True)["n"]) == 0:
    scan_psd()   # ~9 min over the raw csvs
z = np.load(CACHE, allow_pickle=True)
psd = pl.DataFrame({"path": list(z["paths"]), **{f"above30_{c}": z[f"above_{c}"] for c in CHANNELS}})
psd = psd.join(df.select(["path", "dataset", "broken", "original_hz"]), on="path")
print(f"{int(z['n'])} samples scanned")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for c in CHANNELS:
    ax[0].loglog(z["fgrid"][1:], z[f"curve_{c}"][1:], color=C[c], label=c)
ax[0].axvline(30, color="k", ls="--", lw=1)
ax[0].annotate("60 Hz Nyquist", (30, ax[0].get_ylim()[1]), fontsize=8, rotation=90, va="top", ha="right")
ax[0].set(xlabel="frequency (Hz)", ylabel="mean normalised PSD", title="Average spectrum by channel")
ax[0].legend()

for c in CHANNELS:
    v = np.maximum(psd[f"above30_{c}"].to_numpy() * 100, 1e-7)
    ax[1].ecdf(v, color=C[c], label=c, complementary=True)
ax[1].set(xscale="log", yscale="log", xlabel="% of power above 30 Hz",
          ylabel="fraction of samples exceeding", title="Tail: what 60 Hz would discard")
ax[1].axvline(1, color="k", ls=":", lw=1)
ax[1].legend()
fig.tight_layout()

print(f"{'chan':5}{'median':>11}{'p99':>11}{'p99.9':>11}{'max':>11}")
for c in CHANNELS:
    v = psd[f"above30_{c}"].to_numpy() * 100
    print(f"{c:5}{np.median(v):10.5f}%{np.percentile(v,99):10.4f}%{np.percentile(v,99.9):10.4f}%{v.max():10.3f}%")

`pos` and `vel` are clean everywhere. `tau`'s tail is corruption (section 4).
`acc`'s tail is something else: a narrowband 30–40 Hz peak, all in BMLmovi, in samples that are
otherwise fine. Real motion decays monotonically; this does not.

In [ ]:
worst = psd.sort("above30_acc", descending=True).head(1)["path"][0]
typical = "KIT/513/downstairs01_stageii.csv"
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, p, ttl in [(ax[0], worst, "BMLmovi outlier"), (ax[1], typical, "typical (KIT)")]:
    r = df.filter(pl.col("path") == p).row(0, named=True)
    m = _load_sample(RAW_ROOT / p, HUMAN)["motion"][:, :, 2]
    x = m - m.mean(0)
    f, P = signal.welch(x, fs=r["original_hz"], nperseg=min(512, len(x)), axis=0)
    P = P.sum(1) / P.sum()
    a.semilogy(f, P, color="#ee854a")
    a.axvspan(30, r["original_hz"] / 2, color="red", alpha=.12)
    a.axvline(30, color="k", ls="--", lw=1)
    a.set(xlabel="frequency (Hz)", ylabel="normalised PSD",
          title=f"{ttl}\n{p.split('/')[-1][:34]}", xlim=(0, r["original_hz"] / 2))
fig.tight_layout()
print("red band = discarded by 60 Hz resampling")

### Why the anti-alias filter is not optional

Naive decimation (`x[::2]`) folds everything above 30 Hz back into the band we keep.
`resample_poly` filters first, so it removes that energy instead of mirroring it.

In [ ]:
r = df.filter(pl.col("path") == worst).row(0, named=True)
m = _load_sample(RAW_ROOT / worst, HUMAN)["motion"][:, :, 2]
x = m - m.mean(0)
HZ = r["original_hz"]
naive = x[::2]
proper = signal.resample_poly(x, up=1, down=2, axis=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
f0, P0 = signal.welch(x, fs=HZ, nperseg=min(256, len(x)), axis=0)
ax[0].semilogy(f0, P0.sum(1) / P0.sum(), color="grey", label=f"original {HZ:.0f} Hz")
for lab, y, col in [("naive x[::2]", naive, "#d1615d"), ("resample_poly", proper, "#4878a8")]:
    f1, P1 = signal.welch(y, fs=HZ / 2, nperseg=min(256, len(y)), axis=0)
    ax[0].semilogy(f1, P1.sum(1) / P1.sum(), color=col, label=lab)
ax[0].axvline(30, color="k", ls="--", lw=1)
ax[0].set(xlabel="frequency (Hz)", ylabel="normalised PSD", title="Spectrum after downsampling", xlim=(0, 60))
ax[0].legend(fontsize=8)

n0 = 400
tt = np.arange(n0) / HZ
j = np.abs(x).max(0).argmax()
ax[1].plot(tt, x[:n0, j], color="grey", lw=.8, label="original")
ax[1].plot(tt[::2][:n0//2], naive[:n0//2, j], color="#d1615d", lw=1, label="naive")
ax[1].plot(np.arange(len(proper))[:n0//2] / (HZ/2), proper[:n0//2, j], color="#4878a8", lw=1, label="resample_poly")
ax[1].set(xlabel="time (s)", ylabel=f"acc — {DOFS[j]}", title="Waveform", xlim=(0, n0/HZ))
ax[1].legend(fontsize=8)
fig.tight_layout()

## 4. Corrupt torque

The inverse-dynamics solve diverges near singular configurations, producing step discontinuities.
Metric: max |dtau/dt| in N·m/s (rate-normalised so 60 and 120 Hz compare).
Threshold = 5 robust sigmas above the median in log space.

In [ ]:
from sometria.preprocess import TAU_RATE_MAX

lr = np.log10(df["tau_rate"].to_numpy().clip(1))
med, mad = np.median(lr), np.median(np.abs(lr - np.median(lr))) * 1.4826

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(lr, bins=60, color="#4878a8")
ax[0].axvline(np.log10(TAU_RATE_MAX), color="#d1615d", ls="--")
ax[0].annotate(f" threshold\n {TAU_RATE_MAX:.0e}", (np.log10(TAU_RATE_MAX), ax[0].get_ylim()[1]*.7),
               color="#d1615d", fontsize=8)
ax[0].set(xlabel="log10 max |dtau/dt|  (N·m/s)", ylabel="samples",
          title=f"Torque discontinuity\nmedian {10**med:,.0f}, sigma {mad:.2f} dex")

g = (df.group_by("dataset").agg(pl.len().alias("n"), pl.col("broken").sum().alias("bad"))
       .with_columns((100 * pl.col("bad") / pl.col("n")).alias("pct"))
       .filter(pl.col("bad") > 0).sort("pct", descending=True).head(10))
ax[1].barh(g["dataset"].to_list(), g["pct"].to_list(), color="#d1615d")
ax[1].invert_yaxis()
ax[1].set(xlabel="% of samples flagged", title="Corruption by dataset")
for i, (n, b) in enumerate(zip(g["n"], g["bad"])):
    ax[1].text(1, i, f" {b}/{n}", va="center", fontsize=7)

bad_path = df.filter("broken").sort("tau_rate", descending=True)["path"][0]
raw_bad = _load_sample(RAW_ROOT / bad_path, HUMAN)
tau = raw_bad["motion"][:, :, 3]
j = np.abs(np.diff(tau, axis=0)).max(0).argmax()
ax[2].plot(np.arange(len(tau)) / raw_bad["hz"], tau[:, j], lw=.8, color="#d1615d")
ax[2].set(xlabel="time (s)", ylabel=f"tau — {DOFS[j]} (N·m)",
          title=f"Worst sample\n{bad_path.split('/')[-1][:32]}")
fig.tight_layout()

print(df.group_by("split", "broken").len().sort("split", "broken"))
print(f"\nbroken: {df['broken'].sum()}/{len(df)} ({100*df['broken'].mean():.1f}%)")

Corruption is confined to `tau` — acceleration in flagged samples spans the same range as in clean ones.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for lab, f_, col in [("clean", ~pl.col("broken"), "#4878a8"), ("flagged", pl.col("broken"), "#d1615d")]:
    ax[0].ecdf(np.log10(df.filter(f_)["tau_absmax"].to_numpy().clip(1e-3)), label=lab, color=col)
    v = psd.filter(f_)["above30_acc"].to_numpy() * 100
    ax[1].ecdf(np.maximum(v, 1e-7), label=lab, color=col, complementary=True)
ax[0].set(xlabel="log10 peak |tau| (N·m)", ylabel="fraction ≤ x", title="Torque magnitude")
ax[0].legend()
ax[1].set(xscale="log", yscale="log", xlabel="% acc power above 30 Hz",
          ylabel="fraction exceeding", title="Acceleration is unaffected")
ax[1].legend()
fig.tight_layout()

## 5. BABEL splits

BABEL annotates 17 of 26 datasets. Label count grows with duration, which is what makes
sequence-level multi-hot weak on long clips.

In [ ]:
import json
labels = {}
for s in ("train", "val"):
    for v in json.load(open(ROOT / f"splits/babel/{s}.json")).values():
        a = v.get("frame_ann") or v.get("seq_ann")
        cats = {c for l in a["labels"] for c in (l["act_cat"] or []) if c != "transition"}
        labels[v["babel_sid"]] = (v["dur"], len(cats))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sp = df.group_by("split").len().sort("len", descending=True)
names = [s if s else "unlabelled" for s in sp["split"]]
ax[0].bar(names, sp["len"], color=["#bbb" if not s else "#4878a8" for s in sp["split"]])
for i, v in enumerate(sp["len"]):
    ax[0].text(i, v, f" {v}", ha="center", va="bottom", fontsize=8)
ax[0].set(ylabel="samples", title="Split coverage")

du = np.array([v[0] for v in labels.values()])
nl = np.array([v[1] for v in labels.values()])
edges = [0, 5, 10, 20, 60, 1e9]
mids = ["0-5", "5-10", "10-20", "20-60", "60+"]
means = [nl[(du >= a) & (du < b)].mean() for a, b in zip(edges, edges[1:])]
ax[1].bar(mids, means, color="#ee854a")
for i, v in enumerate(means):
    ax[1].text(i, v, f" {v:.1f}", ha="center", va="bottom", fontsize=8)
ax[1].set(xlabel="sequence duration (s)", ylabel="mean distinct action classes", title="Label density grows with length")

cov = (df.group_by("dataset").agg(pl.len().alias("n"), pl.col("split").is_not_null().sum().alias("lab"))
         .with_columns((100 * pl.col("lab") / pl.col("n")).alias("pct")).sort("n", descending=True).head(12))
ax[2].barh(cov["dataset"].to_list(), cov["pct"].to_list(), color="#6acc64")
ax[2].invert_yaxis()
ax[2].set(xlabel="% covered by BABEL", title="Coverage by dataset")
fig.tight_layout()

## 6. Cropping and sampler weighting

One random crop per sample per epoch shows ~half the motion and over-samples short clips.
Weighting the sampler by duration matches crop share to content share.

In [ ]:
dd = df.filter(~pl.col("broken") & (pl.col("duration") >= 1.0))["duration"].to_numpy()
buckets = [(1, 5), (5, 10), (10, 20), (20, 35)]
crop_u = [((dd >= a) & (dd < b)).mean() * 100 for a, b in buckets]
hours = [dd[(dd >= a) & (dd < b)].sum() / dd.sum() * 100 for a, b in buckets]
w = dd / dd.sum()
crop_w = [w[(dd >= a) & (dd < b)].sum() * 100 for a, b in buckets]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
xs = np.arange(len(buckets))
ax[0].bar(xs - .27, crop_u, .27, label="crops (uniform)", color="#d1615d")
ax[0].bar(xs, crop_w, .27, label="crops (duration-weighted)", color="#4878a8")
ax[0].bar(xs + .27, hours, .27, label="share of motion hours", color="#6acc64")
ax[0].set_xticks(xs, [f"{a}-{b}s" for a, b in buckets])
ax[0].set(ylabel="% of epoch", title="Uniform sampling over-weights short clips")
ax[0].legend(fontsize=8)

ws = np.arange(1, 16)
pad_u = [np.maximum(0, W - dd).mean() / W * 100 for W in ws]
pad_w = [(w * np.maximum(0, W - dd)).sum() / W * 100 for W in ws]
ax[1].plot(ws, pad_u, "o-", color="#d1615d", ms=3, label="uniform sampling")
ax[1].plot(ws, pad_w, "o-", color="#4878a8", ms=3, label="duration-weighted")
ax[1].axvline(WINDOW_S, color="k", ls=":", lw=1)
ax[1].set(xlabel="crop window (s)", ylabel="% of window that is zero padding",
          title="Padding waste vs window size")
ax[1].legend(fontsize=8)
fig.tight_layout()

seen = np.minimum(dd, WINDOW_S).sum() / dd.sum() * 100
print(f"uniform, one crop per sample: {seen:.0f}% of motion seen per epoch")
print(f"padding at {WINDOW_S:g}s — uniform {pad_u[int(WINDOW_S)-1]:.1f}%, duration-weighted {pad_w[int(WINDOW_S)-1]:.1f}%")

## Summary

| decision | evidence |
|---|---|
| resample to 60 Hz | `pos` max 0.000%, `vel` max 0.207% of power above 30 Hz |
| use `resample_poly`, never `x[::2]` | BMLmovi carries up to 11% of `acc` power at 30–40 Hz, which aliasing folds back |
| exclude `broken` | 858 samples (4.9%) with non-physical torque discontinuities |
| drop `duration < 1 s` | 59 samples, 0.7 min, up to 99% padding in any window |
| weight sampler by duration | uniform sampling over-weights short clips ~2.4x and pads ~32% of windows |